# Machine Learning Report
## Hotel Occupancy Prediction for Antalya/Alanya Resort Hotels Using Google Trends

**Prepared by:** Bedirhan Sar  
**Project:** Hotel Occupancy Analysis and Prediction

---

## Purpose of the ML Stage

The EDA phase showed that:
- occupancy is strongly seasonal,
- same-day Google Trends relationships are generally weak,
- several lagged Google Trends features are more promising,
- and the strongest signals come mainly from selected Türkiye and Germany keywords.

The purpose of the ML stage is therefore not to replace the EDA logic, but to test a focused predictive question:

> Do lagged Google Trends features improve hotel occupancy prediction beyond hotel identity, calendar seasonality, and recent occupancy history?


## Modeling Strategy Used So Far

Two modeling settings were compared.

### 1. Baseline model
Feature set:
- hotel identity,
- calendar features,
- cyclical seasonality encoding,
- occupancy lags at 7, 14, and 28 days.

This setup captures the main internal structure of the problem: hotel-specific level differences, seasonality, and autoregressive behavior.

### 2. Baseline + lagged Google Trends model
This setup includes everything in the baseline model, plus selected lagged Google Trends variables chosen from the EDA stage.

The intention is to test whether Trends adds **incremental signal** beyond what occupancy history and seasonality already explain.


## Feature Engineering

### Calendar and seasonality features
The following features were used to capture recurring temporal structure:
- `month`
- `day_of_week`
- `week_of_year`
- `doy_sin`
- `doy_cos`

### Autoregressive occupancy features
Recent hotel behavior was represented with:
- `occupancy_lag_7`
- `occupancy_lag_14`
- `occupancy_lag_28`

### Lagged Google Trends features
The strongest feature-lag combinations from the earlier EDA pass were used as a compact first-pass Trends feature set.

Important correction applied in the latest version:

Because Google Trends variables are **date-level shared signals**, their lagged versions must be created on a **unique date table first** and then merged back by date. They should not be shifted directly on the hotel-date modeling table.

This alignment issue has now been corrected in:
- `scripts/modeling_baseline_commented.py`
- `scripts/hotel_normalization_robustness_commented.py`


## Models Compared

Two models were used in the ML comparison:

### Ridge Regression
A regularized linear benchmark.

### Random Forest Regressor
A stronger non-linear benchmark that can model interactions and non-linear relationships without requiring explicit feature transformations.


## Validation Design

The validation strategy was expanded step by step.

### 1. First-pass temporal holdout
The first ML pass used a **time-aware split** instead of random train/test splitting:
- earlier dates were used for training,
- later dates were used for testing.

This was a reasonable starting point, but it still had one methodological weakness: the baseline and baseline + Trends models did not necessarily use the exact same rows before the split.

### 2. Fair same-window comparison
To fix that issue, a second script was added:

- `scripts/modeling_fair_comparison_commented.py`

This version restricts both model settings to the **exact same rows** and then applies **one shared future test window**. This makes the baseline vs baseline + Trends comparison more defensible.

### 3. Walk-forward validation
To move beyond a single holdout period, a third script was added:

- `scripts/modeling_walk_forward_commented.py`

This version uses **expanding-window walk-forward validation**. The model is trained on earlier dates and then tested on the next future block, repeated across several folds. This checks whether the Trends signal is useful **consistently across time**, not just in one final split.


## Current First-Pass ML Findings

After correcting lagged Google Trends alignment and rerunning the first-pass pipeline, the main result remained consistent:

- the best baseline-only model was weaker,
- the best baseline + lagged Trends model performed better,
- and the best overall configuration remained **Random Forest with lagged Google Trends**.

### Updated pooled result
- Best baseline-only RMSE: **approximately 5.87**
- Best baseline + Trends RMSE: **approximately 4.80**

This means that lagged Google Trends still appears to add useful predictive information after the lag-construction bug was corrected.


## Hotel-Level Prediction Plots

These figures show actual vs predicted occupancy for the best non-robustness trends-augmented model.

### Azura Deluxe
![Azura Deluxe prediction](../model_outputs/baseline_ml/actual_vs_pred_azura_deluxe.png)

### Side Mare Hotel
![Side Mare Hotel prediction](../model_outputs/baseline_ml/actual_vs_pred_side_mare_hotel.png)


## Hotel-wise Normalization Robustness in the ML Stage

The EDA stage already tested whether pooled correlation findings survive hotel-wise normalization. The ML stage extends that idea by asking:

> If the target is normalized within each hotel, does the incremental value of lagged Google Trends remain visible?

This matters because the two hotels do not operate at exactly the same occupancy level or variability scale.

### Normalized target used in robustness modeling

`target_hotel_z = (occupancy_rate - train_hotel_mean) / train_hotel_std`

Important leakage rule:
- hotel mean and standard deviation were computed from the **train period only**,
- then applied to both train and test.

This keeps the normalization leakage-safe.

### Robustness result
After rerunning the corrected robustness pipeline:
- the best hotel-normalized baseline-only model remained weaker,
- the best hotel-normalized baseline + Trends model still performed better,
- and the best model again remained **Random Forest with lagged Google Trends**.

### Updated robustness result (raw scale after back-transformation)
- Best baseline-only RMSE: **approximately 6.19**
- Best baseline + Trends RMSE: **approximately 5.67**

This is important because it suggests that the value of lagged Google Trends is **not only an artifact of pooling two hotels with different average occupancy levels**.


## Hotel-normalized Prediction Plots

These figures show actual vs predicted occupancy for the best hotel-normalized robustness model, back-transformed to raw occupancy scale.

### Azura Deluxe
![Azura Deluxe normalized-target robustness prediction](../model_outputs/hotel_normalization_robustness/actual_vs_pred_hotel_z_azura_deluxe.png)

### Side Mare Hotel
![Side Mare Hotel normalized-target robustness prediction](../model_outputs/hotel_normalization_robustness/actual_vs_pred_hotel_z_side_mare_hotel.png)


## Fair Same-Window Validation

The first-pass comparison was useful, but it still allowed the baseline and baseline + Trends models to be built on slightly different non-missing datasets. The fair same-window pass fixes this by forcing both setups to use the **same rows** and the **same test period**.

### Main result
On this cleaner comparison window:
- best baseline-only RMSE: **4.974**
- best baseline + Trends RMSE: **4.798**

The best model again remained **Random Forest**, and the Trends-augmented version still performed better than the baseline-only version.

This matters because it shows that the earlier Trends gain does **not disappear** when the comparison is made on a stricter, methodologically cleaner evaluation window.

### Fair same-window prediction plots

#### Azura Deluxe
![Azura Deluxe fair same-window prediction](../model_outputs/fair_same_window_comparison/actual_vs_pred_same_window_azura_deluxe.png)

#### Side Mare Hotel
![Side Mare Hotel fair same-window prediction](../model_outputs/fair_same_window_comparison/actual_vs_pred_same_window_side_mare_hotel.png)


## Walk-Forward Validation

The walk-forward stage is a stronger time-series validation step than a single holdout. Instead of testing on only one final block, it evaluates the same modeling logic across several future periods.

### Why this matters
A single split can sometimes look unusually good or unusually bad depending on the chosen period. Walk-forward validation checks whether the conclusion is **stable across time**.

### Main result
Across folds, the best average performance again came from the **Random Forest with lagged Google Trends**:

- best baseline-only mean RMSE across folds: **8.166**
- best baseline + Trends mean RMSE across folds: **8.035**

So the average improvement from Trends remained **positive but modest**.

### Interpretation
The walk-forward results are clearly more variable than the single holdout results. Some folds are much harder than others, and overall performance is less stable across time. This suggests that:
- seasonality and timing effects remain dominant,
- the Google Trends contribution is useful but not large,
- and final claims should rely more on repeated validation than on a single test split.

### Walk-forward validation figures

#### RMSE by fold
![Walk-forward RMSE by fold](../model_outputs/walk_forward_validation/walk_forward_rmse_by_fold.png)

#### Mean RMSE summary
![Walk-forward mean RMSE summary](../model_outputs/walk_forward_validation/walk_forward_mean_rmse_summary.png)


## What Has Been Done Correctly

At the current project stage, the ML pipeline already includes several strong design choices:

- time-aware train/test splitting,
- correction of the lagged Google Trends alignment bug,
- separate comparison of baseline vs baseline + Trends,
- hotel-level lag creation for occupancy,
- pooled and hotel-level evaluation,
- robustness analysis with hotel-wise normalized target,
- train-only normalization for the robustness target,
- fair same-window comparison on the same rows and same test period,
- walk-forward validation across multiple future folds,
- back-transformed RMSE reporting for business interpretability.


## Current Limitations of the ML Stage

Although the ML stage is now much stronger than the initial first pass, it is still not the final modeling design.

### 1. Walk-forward performance is still unstable
The walk-forward results vary substantially across folds. This means predictive difficulty changes across time periods, so final claims should stay cautious.

### 2. Benchmark set can still be improved
The current benchmark set includes Ridge and Random Forest. Additional baselines such as naive persistence or a simple seasonal naive model would make the comparison stronger.

### 3. Feature refinement is still incomplete
The current Trends feature set is still a compact first-pass selection. A more systematic feature refinement stage is still needed.

### 4. External signal set is still narrow
Google Trends is useful, but it is still only one external signal family. One carefully chosen external dataset may still improve the project.


## Interpretation

The ML evidence now supports a more methodologically solid conclusion than before:

- **seasonality and recent occupancy remain the dominant drivers**,
- **lagged Google Trends adds incremental predictive value** in the first-pass, fair same-window, and robustness comparisons,
- and under walk-forward validation that value remains present on average, but the gain is **modest** and less stable across time.

So the ML stage is aligned with the EDA logic:
- Google Trends is not a strong same-day demand proxy,
- but selected lagged search signals can act as an **early supporting indicator**,
- especially when combined with seasonality and past occupancy rather than used on their own.


## Recommended Next ML Steps

The next clean steps for the project are:

1. add one or two stronger benchmark baselines such as persistence and seasonal naive,  
2. refine feature selection using only properly aligned lagged features,  
3. inspect feature importance and stability across folds,  
4. decide whether one additional external dataset is worth adding,  
5. summarize final conclusions in a more presentation-ready ML results section.


## Related Files

Main scripts:
- `scripts/modeling_baseline_commented.py`
- `scripts/hotel_normalization_robustness_commented.py`
- `scripts/modeling_fair_comparison_commented.py`
- `scripts/modeling_walk_forward_commented.py`

Main output folders:
- `model_outputs/baseline_ml/`
- `model_outputs/hotel_normalization_robustness/`
- `model_outputs/fair_same_window_comparison/`
- `model_outputs/walk_forward_validation/`

Main report tables:
- `reports/ML_reports/model_comparison.csv`
- `reports/ML_reports/model_comparison_by_hotel.csv`
- `reports/ML_reports/model_comparison_same_window.csv`
- `reports/ML_reports/model_comparison_by_hotel_same_window.csv`
- `reports/ML_reports/walk_forward_fold_results.csv`
- `reports/ML_reports/walk_forward_summary.csv`
- `reports/ML_reports/walk_forward_by_hotel.csv`


## Current Status

**Status:** First-pass ML completed, lagged Google Trends alignment bug corrected, hotel-normalized robustness completed, fair same-window comparison added, and walk-forward validation added.
